# Test stats buttons in TopoEditor
Creates a small flat topo with fake stats injected so the Mean/Max/Min buttons are exercisable without running compute_stats.

In [ ]:
%load_ext autoreload
%autoreload 2
import types
import numpy as np
import xarray as xr
from mom6_forge.grid import Grid
from mom6_forge.topo import Topo
from mom6_forge.topo_editor import TopoEditor

In [2]:
# Small 5x5 grid, flat 1000m topo, no git
grid = Grid(lenx=10.0, leny=10.0, resolution=2.0, xstart=-60.0, ystart=30.0, name="test")
topo = Topo(grid, min_depth=10.0, git=False)
topo.set_flat(1000.0)

In [3]:
# Inject fake stats — shape matches the topo grid (ny, nx)
ny, nx = topo.masked_depth.data.shape
rng = np.random.default_rng(42)

fake_stats = xr.Dataset(
    {
        "OCN_FRAC": xr.DataArray(
            rng.uniform(0.5, 1.0, (ny, nx)),
            dims=["ny", "nx"],
            attrs={"long_name": "ocean fraction", "units": "1"},
        ),
        "D_mean": xr.DataArray(
            rng.uniform(800.0, 1200.0, (ny, nx)),
            dims=["ny", "nx"],
            attrs={"long_name": "mean ocean depth in cell", "units": "m"},
        ),
        "D_min": xr.DataArray(
            rng.uniform(500.0, 799.0, (ny, nx)),
            dims=["ny", "nx"],
            attrs={"long_name": "minimum ocean depth in cell", "units": "m"},
        ),
        "D_max": xr.DataArray(
            rng.uniform(1201.0, 1500.0, (ny, nx)),
            dims=["ny", "nx"],
            attrs={"long_name": "maximum ocean depth in cell", "units": "m"},
        ),
        "D2_mean": xr.DataArray(
            rng.uniform(1e6, 2e6, (ny, nx)),
            dims=["ny", "nx"],
            attrs={"long_name": "mean squared ocean depth", "units": "m2"},
        ),
    }
)

# Attach via a SimpleNamespace so topo.stats returns the dataset
topo.src = types.SimpleNamespace(stats=fake_stats)

print("topo.stats:", topo.stats)
print("Grid shape:", ny, nx)

topo.stats: <xarray.Dataset> Size: 1kB
Dimensions:   (ny: 5, nx: 5)
Dimensions without coordinates: ny, nx
Data variables:
    OCN_FRAC  (ny, nx) float64 200B 0.887 0.7194 0.9293 ... 0.9853 0.9466 0.8892
    D_mean    (ny, nx) float64 200B 877.9 986.7 817.5 ... 915.3 1.073e+03 855.9
    D_min     (ny, nx) float64 200B 559.8 502.2 735.3 ... 755.2 569.9 517.4
    D_max     (ny, nx) float64 200B 1.285e+03 1.289e+03 ... 1.236e+03 1.489e+03
    D2_mean   (ny, nx) float64 200B 1.909e+06 1.7e+06 ... 1.416e+06 1.042e+06
Grid shape: 5 5


In [4]:
%matplotlib widget
TopoEditor(topo)

TopoEditor(children=(VBox(children=(HTML(value='<h2>Topo Editor</h2>'), VBox(children=(VBox(children=(HTML(val…